# 06 · Entrenamiento QLoRA

**Qué contiene.** Ajusta `unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit` con QLoRA (LoRA r = 16, alfa = 16, 4 bits) sobre las trayectorias de entrenamiento. Una época, 100 pasos, lote efectivo de 16, `train_on_responses_only`, con tres familias de datasets apartadas como control.

| | |
|---|---|
| **Runtime** | GPU T4 (Entorno de ejecución → Cambiar tipo de entorno → T4 GPU) |
| **Entradas (`MyDrive/qlora-series-ar/data/`)** | `trayectorias_train_v1.jsonl`, `tools_v1.json` |
| **Salidas** | `qlora_{MODO}/adapter` (el adaptador); con `MODO = "completo"`, `qlora_completo/adapter` |
| **Costo y tiempo** | Sin costo de API. 75,1 minutos en T4 (memoria pico 4,6 GB) |
| **Orden** | Después de `02`; antes de `07` y `08` |
| **En el repositorio** | `entrenamiento.py`, `scripts/entrenar.py` (`--modo datos` verifica la separación sin GPU) |

**Advertencias.**
- `MODO` viene en `"completo"`: ejecutar el notebook entero entrena durante unos 75 minutos. Para probar, usar `"solo_verificar"` o `"prueba_10_pasos"`.
- Una sola semilla (3407): no hay estimación de la variación entre semillas.


In [ ]:
!pip -q install unsloth
print("CELDA: QLORA - ENTRENAMIENTO COMPLETO")
MODO = "completo"            # "solo_verificar" | "prueba_10_pasos" | "completo"
EPOCAS = 1                   # recomendado: 1 (~1,5 h); con 2 serían ~3 h

import os
os.environ["UNSLOTH_DISABLE_CPU_FALLBACK"] = "1"     # que falle en vez de entrenar en CPU
import re, glob, json, time, random, math
from collections import Counter
import torch
assert torch.cuda.is_available(), "No hay GPU: cambiá el tipo de entorno a T4"
print("GPU:", torch.cuda.get_device_name(0))

from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
import bitsandbytes as bnb
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from google.colab import drive

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
MODELO = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"
MAX_SEQ, SEMILLA = 2048, 3407
SALIDA = f"{RUTA}/qlora_{MODO}"          # cada modo usa su propia carpeta
os.makedirs(SALIDA, exist_ok=True)
print("Carpeta de salida:", SALIDA)

# ---------- 1) Modelo base en 4 bits + adaptadores LoRA ----------
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODELO, max_seq_length=MAX_SEQ, load_in_4bit=True)
assert model.max_seq_length == MAX_SEQ, f"El modelo quedó con max_seq_length={model.max_seq_length}"
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEMILLA)
ent = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros entrenables (LoRA): {ent / 1e6:.1f} M | memoria: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
n4 = sum(1 for m in model.modules() if isinstance(m, bnb.nn.Linear4bit))
entrenables = [n for n, p in model.named_parameters() if p.requires_grad]
print("Capas del modelo base en 4 bits:", n4, "| todos los entrenables son de adaptadores LoRA:",
      all("lora_" in n for n in entrenables))
assert n4 > 0, "El modelo base no está cuantizado a 4 bits: no sería QLoRA"

# ---------- 2) Datos: control separado por familia ----------
tools = json.load(open(f"{RUTA}/tools_v1.json"))
T = [json.loads(l) for l in open(f"{RUTA}/trayectorias_train_v1.jsonl")]
cuenta = Counter(t["familia"] for t in T)
fams = sorted(cuenta)
random.Random(SEMILLA).shuffle(fams)
fam_control, n_control = set(), 0
for f in fams:
    if n_control >= 0.05 * len(T):
        break
    if cuenta[f] > 0.08 * len(T):
        continue
    fam_control.add(f)
    n_control += cuenta[f]

def texto(t):
    return tokenizer.apply_chat_template(t["messages"], tools=tools, tokenize=False)

tr = Dataset.from_list([{"text": texto(t)} for t in T if t["familia"] not in fam_control])
ev = Dataset.from_list([{"text": texto(t)} for t in T if t["familia"] in fam_control])
largos = [len(tokenizer(x["text"], add_special_tokens=False)["input_ids"]) for x in tr]
print(f"Entrenamiento: {len(tr)} | control: {len(ev)} | tokens máx: {max(largos)} (límite {MAX_SEQ})")
assert max(largos) <= MAX_SEQ, "Hay ejemplos más largos que la longitud máxima"

# ---------- 3) Entrenador (descarta lo que la versión instalada no acepte y lo informa) ----------
bf16 = torch.cuda.is_bf16_supported()
kw = dict(per_device_train_batch_size=2, gradient_accumulation_steps=8,
          num_train_epochs=EPOCAS, max_steps=10 if MODO == "prueba_10_pasos" else -1,
          learning_rate=2e-4, lr_scheduler_type="linear", warmup_steps=10,
          weight_decay=0.01, optim="adamw_8bit", fp16=not bf16, bf16=bf16,
          logging_steps=10, seed=SEMILLA, output_dir=SALIDA, report_to="none",
          dataset_text_field="text", max_length=MAX_SEQ, save_strategy="steps", save_steps=50,
          save_total_limit=2, eval_strategy="steps", eval_steps=50, per_device_eval_batch_size=2)
descartados, cfg = [], None
for _ in range(8):
    try:
        cfg = SFTConfig(**kw)
        break
    except TypeError as e:
        m = re.search(r"unexpected keyword argument '(\w+)'", str(e))
        if not m:
            raise
        k = m.group(1)
        if k == "max_length":
            kw.pop("max_length")
            kw["max_seq_length"] = MAX_SEQ
        else:
            kw.pop(k)
            descartados.append(k)
assert cfg is not None, "SFTConfig no aceptó la configuración"
print("Parámetros de SFTConfig descartados:", descartados or "ninguno")
trainer = None
for clave in ("processing_class", "tokenizer"):
    try:
        trainer = SFTTrainer(model=model, **{clave: tokenizer}, train_dataset=tr, eval_dataset=ev, args=cfg)
        print("SFTTrainer acepta:", clave)
        break
    except TypeError as e:
        err = e
assert trainer is not None, f"SFTTrainer no aceptó ninguna variante: {err}"
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")

# ---------- 4) ¿Qué tokens se entrenan? ----------
def entrenados(ej):
    return [i for i, l in zip(ej["input_ids"], ej["labels"]) if l != -100]
muestra = [trainer.train_dataset[i] for i in range(min(300, len(trainer.train_dataset)))]
txt = [tokenizer.decode(entrenados(e)) for e in muestra]
frac = sum(len(entrenados(e)) for e in muestra) / sum(len(e["input_ids"]) for e in muestra)
print(f"\nTokens entrenados: {frac:.1%} | ejemplos que entrenan un <tool_response> (debe ser 0): "
      f"{sum('<tool_response>' in t for t in txt)} | llamadas por ejemplo: {dict(Counter(t.count('<tool_call>') for t in txt))}")

# ---------- 5) Entrenamiento ----------
pasos_epoca = math.ceil(len(tr) / 16)
print(f"\nPasos por época: {pasos_epoca} | totales: {EPOCAS * pasos_epoca} | modo: {MODO}")
if MODO in ("prueba_10_pasos", "completo"):
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    retomar = MODO == "completo" and bool(glob.glob(f"{SALIDA}/checkpoint-*"))
    print("Retomando desde un checkpoint:", retomar)
    res = trainer.train(resume_from_checkpoint=retomar)
    seg = time.time() - t0
    print(f"\nTiempo de esta ejecución: {seg / 60:.1f} min | pérdida de entrenamiento: "
          f"{res.metrics.get('train_loss')} | memoria pico: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
    ev_log = [(h["step"], round(h["eval_loss"], 4)) for h in trainer.state.log_history if "eval_loss" in h]
    tr_log = [(h["step"], round(h["loss"], 4)) for h in trainer.state.log_history if "loss" in h]
    print("Pérdida de control (paso, valor):", ev_log)
    print("Pérdida de entrenamiento (paso, valor):", tr_log)
    if MODO == "completo":
        model.save_pretrained(f"{SALIDA}/adapter")
        tokenizer.save_pretrained(f"{SALIDA}/adapter")
        print("Adaptador guardado en:", f"{SALIDA}/adapter")